In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

## Data Pre Processing

In [2]:
device=torch.device("cuda" if torch.cuda.is_available else "cpu")
print(device)
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
    ])
train_dataset=datasets.MNIST(root="data",train=True,transform=transform,download=True)
test_dataset=datasets.MNIST(root="data",train=True,transform=transform,download=True)
train_dataset=torch.utils.data.Subset(train_dataset,range(2000))
test_dataset=torch.utils.data.Subset(test_dataset,range(2000))
train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=64,shuffle=True)
model=models.resnet18(weights=None)
model.load_state_dict(torch.load("data/resnet18-f37072fd.pth",map_location=device))
for parameter in model.parameters():
    parameter.requires_grad=False
for parameter in model.layer4.parameters():
    parameter.requires_grad=True
model.fc=nn.Linear(model.fc.in_features,10)

cuda


## Training

In [6]:
epochs=3
model.to(device)
optimizer=torch.optim.Adam(filter(lambda p:(p.requires_grad),model.parameters()),lr=0.0001)
loss_fn=nn.CrossEntropyLoss()
print("Model Loss:")
for i in range(epochs):
    model.train()
    batch_loss=0
    for data,result in train_loader:
        data=data.to(device)
        result=result.to(device)
        optimizer.zero_grad()
        predicted_result=model(data)
        loss=loss_fn(predicted_result,result)
        loss.backward()
        batch_loss+=loss.item()
        optimizer.step()
    print(f"{i} : {batch_loss/len(train_loader)}")

Model Loss:
0 : 0.002516094402380986
1 : 0.0006606367542190128
2 : 0.00047953731518646237


## Model Testing

In [7]:
model.eval()
correct=0
total=0
with torch.no_grad():
    for data,result in test_loader:
        data=data.to(device)
        result=result.to(device)
        predicted_result=model(data)
        final_prediction=torch.argmax(predicted_result,dim=1)
        correct+=(final_prediction==result).sum().item()
        total+=result.size(0)
    print(f"Test Accuracy : {correct/total}")

Test Accuracy : 1.0
